In [1]:
# ============================================================
# CELL 1: Install required libraries for Google Colab
# ============================================================

!pip -q install datasets transformers accelerate evaluate shap vaderSentiment streamlit pyngrok scikit-learn pandas numpy matplotlib seaborn torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 55.1 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2: Import libraries and set deterministic configuration
# ============================================================

import os
import re
import math
import random
import warnings
import numpy as np
import pandas as pd
import torch
import shap
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Dataset, DatasetDict
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)

warnings.filterwarnings("ignore")

# Fixed seed makes results more reproducible
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Select GPU if available in Colab
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cpu


In [3]:
# ============================================================
# CELL 3: Project configuration
# ============================================================

CONFIG = {
    "dataset_name": "ibm-research/vira-intents-live",
    "model_name": "roberta-base",
    "max_length": 160,
    "num_labels": 3,
    "label_names": ["Low Trust", "Medium Trust", "High Trust"],
    "train_size": 0.80,
    "eval_size": 0.10,
    "test_size": 0.10,
    "epochs": 3,
    "batch_size": 8,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "output_dir": "./improved_trust_roberta",
}

In [4]:
# ============================================================
# CELL 4: Load IBM VIRA intent dataset from Hugging Face
# ============================================================

from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    CONFIG["dataset_name"],
    verification_mode="no_checks"
)

print(dataset)

frames = []

for name, ds in dataset.items():
    df = ds.to_pandas()
    df["original_split"] = name
    frames.append(df)

df_raw = pd.concat(frames, ignore_index=True)

print(df_raw.shape)

README.md:   0%|          | 0.00/455 [00:00<?, ?B/s]

data/train-00000-of-00001-d16551c592a0a1(…):   0%|          | 0.00/240k [00:00<?, ?B/s]

data/validation-00000-of-00001-fd5ce72ea(…):   0%|          | 0.00/108k [00:00<?, ?B/s]

data/test-00000-of-00001-fd5ce72ea3e3378(…):   0%|          | 0.00/106k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7434 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3140 [00:00<?, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 7434
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 3140
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3140
    })
})
(13714, 3)


In [5]:
# ============================================================
# CELL 5: Automatically detect the user text column
# ============================================================

def infer_text_column(dataframe):
    # Candidate names commonly used in conversational datasets
    preferred_names = [
        "text", "utterance", "query", "question", "sentence",
        "message", "user_utterance", "input", "body", "turn"
    ]

    for name in preferred_names:
        if name in dataframe.columns:
            return name

    # If no standard column name exists, choose the object column with longest average text
    object_columns = dataframe.select_dtypes(include=["object"]).columns.tolist()

    if not object_columns:
        raise ValueError("No text-like column found in the dataset.")

    avg_lengths = {}

    for col in object_columns:
        avg_lengths[col] = dataframe[col].astype(str).str.len().mean()

    return max(avg_lengths, key=avg_lengths.get)


TEXT_COLUMN = infer_text_column(df_raw)

print("Detected text column:", TEXT_COLUMN)

df = df_raw[[TEXT_COLUMN]].copy()
df = df.rename(columns={TEXT_COLUMN: "text"})
df["text"] = df["text"].astype(str)
df = df[df["text"].str.strip().str.len() > 0].drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
df.head()

Detected text column: text
Cleaned dataset shape: (10507, 1)


,text
0,After I have been vaccinated is it necessary f...
1,Should I wear masks after getting vaccinated?
2,How long should I use the mask?
3,Now that I've been vaccinated should I still t...
4,if i recieved the vaccine will i be safe?


In [6]:
# ============================================================
# CELL 6: Text cleaning for trust signal detection
# ============================================================

def clean_text(text):
    # Lowercase text for consistent matching
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Replace contractions important for negation handling
    contractions = {
        "don't": "do not",
        "doesn't": "does not",
        "didn't": "did not",
        "can't": "cannot",
        "couldn't": "could not",
        "won't": "will not",
        "wouldn't": "would not",
        "shouldn't": "should not",
        "isn't": "is not",
        "aren't": "are not",
        "wasn't": "was not",
        "weren't": "were not",
        "haven't": "have not",
        "hasn't": "has not",
        "hadn't": "had not",
        "i'm": "i am",
        "it's": "it is",
        "that's": "that is",
    }

    for old, new in contractions.items():
        text = text.replace(old, new)

    # Keep letters, numbers, apostrophes and useful punctuation
    text = re.sub(r"[^a-z0-9\s\.\,\?\!']", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text"]].head()

,text,clean_text
0,After I have been vaccinated is it necessary f...,after i have been vaccinated is it necessary f...
1,Should I wear masks after getting vaccinated?,should i wear masks after getting vaccinated?
2,How long should I use the mask?,how long should i use the mask?
3,Now that I've been vaccinated should I still t...,now that i've been vaccinated should i still t...
4,if i recieved the vaccine will i be safe?,if i recieved the vaccine will i be safe?


In [7]:
# ============================================================
# CELL 7: Build expanded healthcare trust lexicons
# ============================================================

LEXICONS = {
    "trust_positive": {
        r"\bi trust\b": 2.8,
        r"\btrusted\b": 2.4,
        r"\bcan trust\b": 2.8,
        r"\brely on\b": 2.4,
        r"\bdepend on\b": 2.0,
        r"\bcredible\b": 2.2,
        r"\breliable\b": 2.4,
        r"\baccurate\b": 2.2,
        r"\bcorrect\b": 1.7,
        r"\bclear\b": 1.4,
        r"\bmakes sense\b": 2.2,
        r"\bhelpful\b": 1.8,
        r"\breassuring\b": 2.4,
        r"\bconfident\b": 2.4,
        r"\bi understand\b": 1.8,
        r"\bthank you\b": 1.3,
        r"\bthanks\b": 1.0,
        r"\bgood explanation\b": 2.0,
    },

    "healthcare_trust": {
        r"\bdoctor explained\b": 3.0,
        r"\bclear diagnosis\b": 3.0,
        r"\baccurate diagnosis\b": 3.2,
        r"\bmedical advice makes sense\b": 2.8,
        r"\bthat diagnosis makes sense\b": 2.8,
        r"\bthe doctor is right\b": 2.5,
        r"\bthe information is clear\b": 2.3,
        r"\bthis is reassuring\b": 2.5,
        r"\bi feel reassured\b": 2.6,
        r"\bhelped me understand\b": 2.4,
        r"\bexplained clearly\b": 2.5,
        r"\bprofessional advice\b": 2.0,
    },

    "distrust_negative": {
        r"\bi do not trust\b": 4.5,
        r"\bdonot trust\b": 4.5,
        r"\bdo not trust\b": 4.5,
        r"\bdoes not trust\b": 4.0,
        r"\bcannot trust\b": 4.5,
        r"\bcan not trust\b": 4.5,
        r"\bdon't trust\b": 4.5,
        r"\bcan't trust\b": 4.5,
        r"\buntrusted\b": 3.6,
        r"\bunreliable\b": 3.8,
        r"\bnot reliable\b": 4.0,
        r"\bnot convinced\b": 4.0,
        r"\bdo not believe\b": 3.8,
        r"\bcannot rely\b": 4.0,
        r"\bwrong\b": 2.7,
        r"\bincorrect\b": 2.8,
        r"\bfalse\b": 2.8,
        r"\bmisleading\b": 3.5,
        r"\bsuspicious\b": 3.2,
        r"\bunsafe\b": 3.8,
        r"\bscam\b": 4.0,
        r"\bfake\b": 3.5,
        r"\bdoes not sound right\b": 3.8,
        r"\bdoesn't sound right\b": 3.8,
    },

    "healthcare_distrust": {
        r"\bwrong diagnosis\b": 4.5,
        r"\bmisdiagnosis\b": 4.8,
        r"\bincorrect diagnosis\b": 4.5,
        r"\bunsafe treatment\b": 4.5,
        r"\bunsafe vaccine\b": 4.0,
        r"\bsecond opinion\b": 3.0,
        r"\bmedical error\b": 4.2,
        r"\bbad advice\b": 3.8,
        r"\bconfusing advice\b": 3.3,
        r"\bdoctor is wrong\b": 4.0,
        r"\bnot medically safe\b": 4.5,
        r"\brisky\b": 3.0,
        r"\bharmful\b": 4.0,
        r"\bside effects were ignored\b": 4.2,
    },

    "uncertainty": {
        r"\bnot sure\b": 4.2,
        r"\bunsure\b": 3.8,
        r"\buncertain\b": 3.8,
        r"\bi am not sure\b": 4.3,
        r"\bi don't know\b": 3.2,
        r"\bi do not know\b": 3.2,
        r"\bhard to say\b": 3.4,
        r"\bunclear\b": 3.2,
        r"\bconfused\b": 3.5,
        r"\bmaybe\b": 2.3,
        r"\bperhaps\b": 2.3,
        r"\bpossibly\b": 2.5,
        r"\bprobably\b": 1.5,
        r"\bseems\b": 1.6,
        r"\bappears\b": 1.6,
        r"\bapparently\b": 2.2,
        r"\bmight\b": 2.4,
        r"\bcould\b": 1.8,
        r"\bmay\b": 1.6,
        r"\bi wonder\b": 2.7,
    },

    "hedging": {
        r"\bi think\b": 2.2,
        r"\bi guess\b": 2.8,
        r"\bi suppose\b": 2.7,
        r"\bit seems\b": 2.2,
        r"\bit appears\b": 2.1,
        r"\bas far as i know\b": 2.0,
        r"\bkind of\b": 1.7,
        r"\bsort of\b": 1.7,
        r"\bmore or less\b": 1.8,
        r"\bnot completely\b": 2.3,
        r"\bnot entirely\b": 2.3,
    },

    "skepticism": {
        r"\bare you sure\b": 3.5,
        r"\bis that true\b": 3.3,
        r"\bprove\b": 2.8,
        r"\bevidence\b": 1.8,
        r"\bsource\b": 1.5,
        r"\bwhere did you get\b": 3.0,
        r"\bhow do you know\b": 3.3,
        r"\bi doubt\b": 3.8,
        r"\bdoubtful\b": 3.5,
        r"\bskeptical\b": 3.5,
        r"\bnot buying\b": 3.5,
    },

    "certainty": {
        r"\bi am sure\b": 3.0,
        r"\bdefinitely\b": 2.8,
        r"\bcertainly\b": 2.6,
        r"\bclearly\b": 2.1,
        r"\babsolutely\b": 2.4,
        r"\bwithout doubt\b": 3.2,
        r"\bno doubt\b": 3.0,
        r"\bconfirmed\b": 2.4,
        r"\bverified\b": 2.5,
    },

    "reassurance": {
        r"\breassured\b": 2.8,
        r"\brelieved\b": 2.0,
        r"\bcomforting\b": 2.2,
        r"\bhelped me\b": 1.8,
        r"\bfeel better\b": 2.0,
        r"\bclear now\b": 2.1,
    }
}

In [8]:
# ============================================================
# CELL 8: Negation-aware weighted feature extraction
# ============================================================

NEGATION_TERMS = [
    "not", "no", "never", "none", "cannot", "can't", "do not", "does not",
    "did not", "will not", "would not", "is not", "are not", "was not",
    "were not", "without"
]

POSITIVE_ANCHORS = [
    "trust", "trusted", "reliable", "credible", "accurate", "correct",
    "safe", "helpful", "clear", "reassuring", "confident", "believe", "rely"
]

NEGATIVE_ANCHORS = [
    "wrong", "incorrect", "unsafe", "unreliable", "misleading", "fake",
    "false", "confused", "doubt", "skeptical", "misdiagnosis"
]


def weighted_regex_score(text, pattern_weights):
    # Returns total weighted score and matched phrases
    total = 0.0
    matches = []

    for pattern, weight in pattern_weights.items():
        found = re.findall(pattern, text)

        if found:
            count = len(found)
            total += weight * count
            matches.append(pattern.replace(r"\b", "").replace("\\", ""))

    return total, matches


def detect_negated_positive(text, window=4):
    # Strongly penalize positive trust words when a negation appears nearby
    tokens = text.split()
    score = 0.0
    matches = []

    for i, token in enumerate(tokens):
        if token in POSITIVE_ANCHORS:
            start = max(0, i - window)
            context = " ".join(tokens[start:i])

            if any(neg in context for neg in NEGATION_TERMS):
                score += 3.8
                matches.append("negated_" + token)

    return score, matches


def extract_trust_features(text):
    # Clean the input before extracting all signals
    text = clean_text(text)

    trust_positive, trust_positive_matches = weighted_regex_score(text, LEXICONS["trust_positive"])
    healthcare_trust, healthcare_trust_matches = weighted_regex_score(text, LEXICONS["healthcare_trust"])
    distrust_negative, distrust_negative_matches = weighted_regex_score(text, LEXICONS["distrust_negative"])
    healthcare_distrust, healthcare_distrust_matches = weighted_regex_score(text, LEXICONS["healthcare_distrust"])
    uncertainty, uncertainty_matches = weighted_regex_score(text, LEXICONS["uncertainty"])
    hedging, hedging_matches = weighted_regex_score(text, LEXICONS["hedging"])
    skepticism, skepticism_matches = weighted_regex_score(text, LEXICONS["skepticism"])
    certainty, certainty_matches = weighted_regex_score(text, LEXICONS["certainty"])
    reassurance, reassurance_matches = weighted_regex_score(text, LEXICONS["reassurance"])
    negated_positive, negated_positive_matches = detect_negated_positive(text)

    sentiment = SentimentIntensityAnalyzer().polarity_scores(text)["compound"]

    # Confidence is not sentiment; it is certainty minus uncertainty and hedging
    confidence_score = 50 + (certainty * 8) + (trust_positive * 2) - (uncertainty * 8) - (hedging * 6) - (skepticism * 5)
    confidence_score = float(np.clip(confidence_score, 0, 100))

    feature_dict = {
        "trust_positive": trust_positive,
        "healthcare_trust": healthcare_trust,
        "distrust_negative": distrust_negative,
        "healthcare_distrust": healthcare_distrust,
        "uncertainty": uncertainty,
        "hedging": hedging,
        "skepticism": skepticism,
        "certainty": certainty,
        "reassurance": reassurance,
        "negated_positive": negated_positive,
        "sentiment": sentiment,
        "confidence_score": confidence_score,
        "trust_positive_matches": trust_positive_matches,
        "healthcare_trust_matches": healthcare_trust_matches,
        "distrust_negative_matches": distrust_negative_matches,
        "healthcare_distrust_matches": healthcare_distrust_matches,
        "uncertainty_matches": uncertainty_matches,
        "hedging_matches": hedging_matches,
        "skepticism_matches": skepticism_matches,
        "certainty_matches": certainty_matches,
        "reassurance_matches": reassurance_matches,
        "negated_positive_matches": negated_positive_matches,
    }

    return feature_dict


features = df["clean_text"].apply(extract_trust_features)
feature_df = pd.DataFrame(features.tolist())

df = pd.concat([df, feature_df], axis=1)

df.head()

,text,clean_text,trust_positive,healthcare_trust,distrust_negative,healthcare_distrust,uncertainty,hedging,skepticism,certainty,...,trust_positive_matches,healthcare_trust_matches,distrust_negative_matches,healthcare_distrust_matches,uncertainty_matches,hedging_matches,skepticism_matches,certainty_matches,reassurance_matches,negated_positive_matches
0,After I have been vaccinated is it necessary f...,after i have been vaccinated is it necessary f...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,[],[],[],[],[],[],[],[],[],[]
1,Should I wear masks after getting vaccinated?,should i wear masks after getting vaccinated?,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,[],[],[],[],[],[],[],[],[],[]
2,How long should I use the mask?,how long should i use the mask?,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,[],[],[],[],[],[],[],[],[],[]
3,Now that I've been vaccinated should I still t...,now that i've been vaccinated should i still t...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,[],[],[],[],[],[],[],[],[],[]
4,if i recieved the vaccine will i be safe?,if i recieved the vaccine will i be safe?,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,[],[],[],[],[],[],[],[],[],[]


In [9]:
# ============================================================
# CELL 9: Generate improved pseudo trust labels
# ============================================================

def calculate_rule_trust_score(row):
    # Balanced scoring model where positive and negative evidence compete
    baseline = 50.0

    positive = (
        row["trust_positive"] * 5.0 +
        row["healthcare_trust"] * 5.5 +
        row["certainty"] * 3.0 +
        row["reassurance"] * 3.5
    )

    negative = (
        row["distrust_negative"] * 7.0 +
        row["healthcare_distrust"] * 7.5 +
        row["uncertainty"] * 6.5 +
        row["hedging"] * 5.5 +
        row["skepticism"] * 6.0 +
        row["negated_positive"] * 7.0
    )

    # Sentiment is only a small supporting signal
    sentiment_support = row["sentiment"] * 4.0

    raw_score = baseline + positive - negative + sentiment_support

    return float(np.clip(raw_score, 0, 100))


def trust_label_from_score(score):
    # Stricter thresholds prevent uncertain text from becoming High Trust
    if score < 35:
        return 0
    elif score < 70:
        return 1
    else:
        return 2


df["rule_trust_score"] = df.apply(calculate_rule_trust_score, axis=1)
df["label"] = df["rule_trust_score"].apply(trust_label_from_score)
df["trust_label"] = df["label"].map({0: "Low Trust", 1: "Medium Trust", 2: "High Trust"})

print(df["trust_label"].value_counts())
df[["text", "rule_trust_score", "trust_label"]].sample(10, random_state=SEED)

trust_label
Medium Trust    10043
Low Trust         463
High Trust          1
Name: count, dtype: int64


,text,rule_trust_score,trust_label
8985,Could this vaccine work with new phases of the...,38.3000,Medium Trust
565,I dont know if I will qualify for a vaccine?,50.0000,Medium Trust
5173,can i die faster because i am a woman?,47.6024,Medium Trust
3884,What will be the effect on my pre-existing con...,50.0000,Medium Trust
7823,If I have had the vaccine can I carry on as no...,50.0000,Medium Trust
39,Should I continue to use the mask after vaccin...,50.0000,Medium Trust
4614,"To maintain good health, are these security me...",52.5944,Medium Trust
35,Will bio safety measures be necessary even if ...,51.6860,Medium Trust
7680,When can I get vaccinated?,50.0000,Medium Trust
10333,How do I know if the clinical trials were appr...,51.6860,Medium Trust


In [10]:
# ============================================================
# CELL 10: Validate difficult examples before training
# ============================================================

test_examples = [
    "I'm not sure if this trust detection system will work properly.",
    "I do not trust this diagnosis and I want a second opinion.",
    "The doctor explained everything clearly and now I understand.",
    "This advice seems confusing and does not sound right.",
    "Thank you doctor, that makes sense and feels reassuring.",
    "I guess it could be correct but I am not convinced."
]

for example in test_examples:
    temp = extract_trust_features(example)
    temp_score = calculate_rule_trust_score(pd.Series(temp))
    temp_label = CONFIG["label_names"][trust_label_from_score(temp_score)]

    print("\nText:", example)
    print("Score:", round(temp_score, 2))
    print("Label:", temp_label)
    print("Uncertainty:", temp["uncertainty_matches"])
    print("Distrust:", temp["distrust_negative_matches"])
    print("Hedging:", temp["hedging_matches"])


Text: I'm not sure if this trust detection system will work properly.
Score: 0.0
Label: Low Trust
Uncertainty: ['not sure', 'i am not sure']
Distrust: []
Hedging: []

Text: I do not trust this diagnosis and I want a second opinion.
Score: 0.0
Label: Low Trust
Uncertainty: []
Distrust: ['i do not trust', 'do not trust']
Hedging: []

Text: The doctor explained everything clearly and now I understand.
Score: 83.41
Label: High Trust
Uncertainty: []
Distrust: []
Hedging: []

Text: This advice seems confusing and does not sound right.
Score: 12.09
Label: Low Trust
Uncertainty: ['seems']
Distrust: ['does not sound right']
Hedging: []

Text: Thank you doctor, that makes sense and feels reassuring.
Score: 82.05
Label: High Trust
Uncertainty: []
Distrust: []
Hedging: []

Text: I guess it could be correct but I am not convinced.
Score: 1.65
Label: Low Trust
Uncertainty: ['could']
Distrust: ['not convinced']
Hedging: ['i guess']


In [11]:
# ============================================================
# CELL 11: Train-validation-test split
# ============================================================

# Separate the 'High Trust' samples (label=2) as there's only one
df_high_trust = df[df["label"] == 2]
df_other_trust = df[df["label"] != 2]

# Split the remaining data (Low and Medium Trust) into train and temp sets
train_other_df, temp_other_df = train_test_split(
    df_other_trust,
    train_size=CONFIG["train_size"],
    stratify=df_other_trust["label"],
    random_state=SEED
)

# Add the high trust samples to the training set
train_df = pd.concat([train_other_df, df_high_trust])

# Split the temp_other_df into eval and test sets
relative_eval_size = CONFIG["eval_size"] / (CONFIG["eval_size"] + CONFIG["test_size"])

eval_df, test_df = train_test_split(
    temp_other_df,
    train_size=relative_eval_size,
    stratify=temp_other_df["label"],
    random_state=SEED
)

print("Train:", train_df.shape)
print("Eval:", eval_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["trust_label"].value_counts())
print("\nEval labels:")
print(eval_df["trust_label"].value_counts())
print("\nTest labels:")
print(test_df["trust_label"].value_counts())

Train: (8405, 27)
Eval: (1051, 27)
Test: (1051, 27)

Train labels:
trust_label
Medium Trust    8034
Low Trust        370
High Trust         1
Name: count, dtype: int64

Eval labels:
trust_label
Medium Trust    1005
Low Trust         46
Name: count, dtype: int64

Test labels:
trust_label
Medium Trust    1004
Low Trust         47
Name: count, dtype: int64


In [12]:
# ============================================================
# CELL 12: Baseline traditional ML model using TF-IDF
# ============================================================

tfidf = TfidfVectorizer(
    max_features=12000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(train_df["clean_text"])
X_test_tfidf = tfidf.transform(test_df["clean_text"])

ml_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED
)

ml_model.fit(X_train_tfidf, train_df["label"])

ml_preds = ml_model.predict(X_test_tfidf)

print("Traditional ML Accuracy:", accuracy_score(test_df["label"], ml_preds))
print("Traditional ML Macro F1:", f1_score(test_df["label"], ml_preds, average="macro"))
print(classification_report(
    test_df["label"],
    ml_preds,
    labels=ml_model.classes_, # Specify all classes the model was trained on
    target_names=CONFIG["label_names"],
    zero_division='warn'
))

Traditional ML Accuracy: 0.9295908658420552
Traditional ML Macro F1: 0.7009152438086448
              precision    recall  f1-score   support

   Low Trust       0.34      0.62      0.44        47
Medium Trust       0.98      0.94      0.96      1004
  High Trust       0.00      0.00      0.00         0

    accuracy                           0.93      1051
   macro avg       0.44      0.52      0.47      1051
weighted avg       0.95      0.93      0.94      1051



In [13]:
# ============================================================
# CELL 13: Prepare Hugging Face datasets for RoBERTa
# ============================================================

train_dataset = Dataset.from_pandas(train_df[["clean_text", "label"]].reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df[["clean_text", "label"]].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[["clean_text", "label"]].reset_index(drop=True))

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": eval_dataset,
    "test": test_dataset
})

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])


def tokenize_batch(batch):
    # Tokenize text for RoBERTa
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        max_length=CONFIG["max_length"]
    )


tokenized_dataset = dataset_dict.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_dataset

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/8405 [00:00<?, ? examples/s]

Map:   0%|          | 0/1051 [00:00<?, ? examples/s]

Map:   0%|          | 0/1051 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['clean_text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 8405
    })
    validation: Dataset({
        features: ['clean_text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1051
    })
    test: Dataset({
        features: ['clean_text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1051
    })
})

In [14]:
# ============================================================
# CELL 14: Load RoBERTa sequence classification model
# ============================================================

id2label = {0: "Low Trust", 1: "Medium Trust", 2: "High Trust"}
label2id = {"Low Trust": 0, "Medium Trust": 1, "High Trust": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=CONFIG["num_labels"],
    id2label=id2label,
    label2id=label2id
)

model.to(DEVICE)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [15]:
# ============================================================
# CELL 15: Define evaluation metrics
# ============================================================

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    # Compute accuracy and macro F1 for trust labels
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

In [ ]:
# ============================================================
# CELL 16: Fine-tune RoBERTa
# ============================================================

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    num_train_epochs=CONFIG["epochs"],
    weight_decay=CONFIG["weight_decay"],
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
test_predictions = trainer.predict(tokenized_dataset["test"])

test_logits = test_predictions.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()
test_pred_labels = np.argmax(test_probs, axis=1)

print("RoBERTa Test Accuracy:", accuracy_score(test_df["label"], test_pred_labels))
print("RoBERTa Test Macro F1:", f1_score(test_df["label"], test_pred_labels, average="macro"))

# Get all possible label IDs (0, 1, 2) from id2label
all_label_ids = sorted(id2label.keys())

print(classification_report(
    test_df["label"],
    test_pred_labels,
    labels=all_label_ids,  # Explicitly specify all possible labels [0, 1, 2]
    target_names=CONFIG["label_names"], # Use all three target names
    zero_division='warn'
))

# Also pass all_label_ids to confusion_matrix to ensure it includes all classes
cm = confusion_matrix(test_df["label"], test_pred_labels, labels=all_label_ids)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CONFIG["label_names"], yticklabels=CONFIG["label_names"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("RoBERTa Trust Detection Confusion Matrix")
plt.show()

In [ ]:
# ============================================================
# CELL 18: Save trained model and tokenizer
# ============================================================

trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print("Model saved to:", CONFIG["output_dir"])

In [ ]:
# ============================================================
# CELL 19: RoBERTa probability, entropy and calibrated confidence
# ============================================================

trust_classifier = pipeline(
    "text-classification",
    model=CONFIG["output_dir"],
    tokenizer=CONFIG["output_dir"],
    top_k=None,
    device=0 if DEVICE == "cuda" else -1,
)


def get_roberta_outputs(text):
    # Return Low, Medium, High probabilities plus entropy and confidence.
    raw_scores = trust_classifier(text)

    # Handles different Transformers output formats.
    if isinstance(raw_scores, list) and raw_scores and isinstance(raw_scores[0], list):
        scores = raw_scores[0]
    elif isinstance(raw_scores, list) and raw_scores and isinstance(raw_scores[0], dict):
        scores = raw_scores
    elif isinstance(raw_scores, dict):
        scores = [raw_scores]
    else:
        raise TypeError(f"Unexpected pipeline output format: {type(raw_scores)} -> {raw_scores}")

    score_dict = {item["label"]: item["score"] for item in scores}

    low = score_dict.get("Low Trust", score_dict.get("LABEL_0", 0.0))
    medium = score_dict.get("Medium Trust", score_dict.get("LABEL_1", 0.0))
    high = score_dict.get("High Trust", score_dict.get("LABEL_2", 0.0))

    probs = np.array([low, medium, high], dtype=float)

    if probs.sum() == 0:
        probs = np.array([1/3, 1/3, 1/3], dtype=float)
    else:
        probs = probs / probs.sum()

    entropy = entropy_from_probs(probs)

    return {
        "low_prob": float(probs[0]),
        "medium_prob": float(probs[1]),
        "high_prob": float(probs[2]),
        "entropy": entropy,
        "model_confidence": float(1.0 - entropy),
        "expected_trust": float((probs[2] * 100) + (probs[1] * 50)),
        "predicted_label": CONFIG["label_names"][int(np.argmax(probs))],
    }

In [ ]:
# ============================================================
# CELL 20: Final composite trust scoring system
# ============================================================

def final_trust_score(text):
    # Combine RoBERTa, confidence, healthcare trust, uncertainty and negative signals
    features = extract_trust_features(text)
    roberta = get_roberta_outputs(text)

    baseline = 45.0

    roberta_contribution = (roberta["expected_trust"] - 50) * 0.45
    confidence_contribution = (features["confidence_score"] - 50) * 0.20

    positive_contribution = (
        features["healthcare_trust"] * 4.5 +
        features["trust_positive"] * 3.0 +
        features["certainty"] * 2.0 +
        features["reassurance"] * 2.5
    )

    negative_contribution = (
        features["healthcare_distrust"] * 5.5 +
        features["distrust_negative"] * 5.0 +
        features["uncertainty"] * 4.8 +
        features["hedging"] * 3.8 +
        features["skepticism"] * 4.2 +
        features["negated_positive"] * 5.5
    )

    entropy_penalty = roberta["entropy"] * 12.0

    sentiment_support = features["sentiment"] * 3.0

    score = (
        baseline
        + roberta_contribution
        + confidence_contribution
        + positive_contribution
        - negative_contribution
        - entropy_penalty
        + sentiment_support
    )

    score = float(np.clip(score, 0, 100))

    if score < 35:
        label = "Low Trust"
    elif score < 70:
        label = "Medium Trust"
    else:
        label = "High Trust"

    explanation = {
        "Trust Score": score,
        "Trust Label": label,
        "RoBERTa Low Probability": roberta["low_prob"],
        "RoBERTa Medium Probability": roberta["medium_prob"],
        "RoBERTa High Probability": roberta["high_prob"],
        "Model Confidence": roberta["model_confidence"],
        "Prediction Entropy": roberta["entropy"],
        "Confidence Score": features["confidence_score"],
        "Positive Contribution": positive_contribution,
        "Negative Contribution": negative_contribution,
        "Entropy Penalty": entropy_penalty,
        "Sentiment Support": sentiment_support,
        "Healthcare Trust Signals": features["healthcare_trust_matches"],
        "Healthcare Distrust Signals": features["healthcare_distrust_matches"],
        "Trust Signals": features["trust_positive_matches"],
        "Distrust Signals": features["distrust_negative_matches"],
        "Uncertainty Signals": features["uncertainty_matches"],
        "Hedging Signals": features["hedging_matches"],
        "Skepticism Signals": features["skepticism_matches"],
        "Negated Positive Signals": features["negated_positive_matches"],
        "Certainty Signals": features["certainty_matches"],
        "Reassurance Signals": features["reassurance_matches"],
    }

    return explanation

In [ ]:
# ============================================================
# CELL 21: Test final prediction function
# ============================================================

sample_inputs = [
    "I'm not sure if this trust detection system will work properly.",
    "I do not trust this diagnosis and I want a second opinion.",
    "The doctor explained everything clearly and now I understand.",
    "Thank you doctor, that makes sense and feels reassuring.",
    "I guess it could be correct but I am not convinced.",
]

for text in sample_inputs:
    result = final_trust_score(text)

    print("\nInput:", text)
    print("Trust Score:", round(result["Trust Score"], 2))
    print("Trust Label:", result["Trust Label"])
    print("RoBERTa Low:", round(result["RoBERTa Low Probability"], 3))
    print("RoBERTa Medium:", round(result["RoBERTa Medium Probability"], 3))
    print("RoBERTa High:", round(result["RoBERTa High Probability"], 3))
    print("Model Confidence:", round(result["Model Confidence"], 3))
    print("Entropy:", round(result["Prediction Entropy"], 3))
    print("Uncertainty:", result["Uncertainty Signals"])
    print("Distrust:", result["Distrust Signals"])
    print("Hedging:", result["Hedging Signals"])

In [ ]:
# ============================================================
# CELL 22: SHAP explainability for the traditional ML baseline
# ============================================================

# SHAP is used here on the TF-IDF Logistic Regression model because it is faster and stable in Colab
explainer = shap.LinearExplainer(
    ml_model,
    X_train_tfidf,
    feature_perturbation="interventional"
)

sample_size = min(100, X_test_tfidf.shape[0])
X_sample = X_test_tfidf[:sample_size]

shap_values = explainer.shap_values(X_sample)
feature_names = tfidf.get_feature_names_out()

print("SHAP explanation generated for baseline ML model.")
print("X_sample shape:", X_sample.shape)
print("SHAP values type:", type(shap_values))

# Convert sparse TF-IDF sample to dense for plotting
X_sample_dense = X_sample.toarray()

# Handle both old and new SHAP output formats
if isinstance(shap_values, list):
    # Old SHAP format: list of arrays, one per class
    shap_values_by_class = shap_values
else:
    # New SHAP format: array with shape (samples, features, classes)
    if shap_values.ndim == 3:
        shap_values_by_class = [
            shap_values[:, :, class_index]
            for class_index in range(shap_values.shape[2])
        ]
    else:
        # Binary or single-output format
        shap_values_by_class = [shap_values]

# Summary plot for each class
for class_index, class_name in enumerate(CONFIG["label_names"]):
    if class_index >= len(shap_values_by_class):
        print(f"Skipping {class_name}: no SHAP values available for this class.")
        continue

    print("SHAP summary for:", class_name)

    shap.summary_plot(
        shap_values_by_class[class_index],
        X_sample_dense,
        feature_names=feature_names,
        max_display=20
    )

In [ ]:
# ============================================================
# CELL 23: Local explanation helper for one input
# ============================================================

def explain_feature_contributions(text):
    # This gives transparent non-SHAP feature contributions for the final score
    result = final_trust_score(text)

    contribution_rows = [
        ("Positive trust signals", result["Positive Contribution"]),
        ("Negative trust signals", -result["Negative Contribution"]),
        ("Entropy penalty", -result["Entropy Penalty"]),
        ("Sentiment support", result["Sentiment Support"]),
        ("Model confidence", (result["Confidence Score"] - 50) * 0.20),
    ]

    contribution_df = pd.DataFrame(contribution_rows, columns=["Feature Group", "Contribution"])

    plt.figure(figsize=(8, 4))
    sns.barplot(data=contribution_df, x="Contribution", y="Feature Group", palette="coolwarm")
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Feature Contributions to Final Trust Score")
    plt.show()

    return result, contribution_df


result, contribution_df = explain_feature_contributions("I'm not sure if this trust detection system will work properly.")
print(result)
contribution_df

In [ ]:
%%writefile app.py
import re
import numpy as np
import pandas as pd
import streamlit as st
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


MODEL_DIR = "./improved_trust_roberta"
LABEL_NAMES = ["Low Trust", "Medium Trust", "High Trust"]


LEXICONS = {
    "trust_positive": {
        r"\bi trust\b": 2.8,
        r"\btrusted\b": 2.4,
        r"\bcan trust\b": 2.8,
        r"\brely on\b": 2.4,
        r"\bdepend on\b": 2.0,
        r"\bcredible\b": 2.2,
        r"\breliable\b": 2.4,
        r"\baccurate\b": 2.2,
        r"\bcorrect\b": 1.7,
        r"\bclear\b": 1.4,
        r"\bmakes sense\b": 2.2,
        r"\bhelpful\b": 1.8,
        r"\breassuring\b": 2.4,
        r"\bconfident\b": 2.4,
        r"\bi understand\b": 1.8,
        r"\bthank you\b": 1.3,
        r"\bthanks\b": 1.0,
        r"\bgood explanation\b": 2.0,
    },
    "healthcare_trust": {
        r"\bdoctor explained\b": 3.0,
        r"\bclear diagnosis\b": 3.0,
        r"\baccurate diagnosis\b": 3.2,
        r"\bthat diagnosis makes sense\b": 2.8,
        r"\bthe doctor is right\b": 2.5,
        r"\bthe information is clear\b": 2.3,
        r"\bthis is reassuring\b": 2.5,
        r"\bi feel reassured\b": 2.6,
        r"\bhelped me understand\b": 2.4,
        r"\bexplained clearly\b": 2.5,
    },
    "distrust_negative": {
        r"\bi do not trust\b": 4.5,
        r"\bdo not trust\b": 4.5,
        r"\bcannot trust\b": 4.5,
        r"\bcan not trust\b": 4.5,
        r"\bdon't trust\b": 4.5,
        r"\bcan't trust\b": 4.5,
        r"\buntrusted\b": 3.6,
        r"\bunreliable\b": 3.8,
        r"\bnot reliable\b": 4.0,
        r"\bnot convinced\b": 4.0,
        r"\bdo not believe\b": 3.8,
        r"\bcannot rely\b": 4.0,
        r"\bwrong\b": 2.7,
        r"\bincorrect\b": 2.8,
        r"\bfalse\b": 2.8,
        r"\bmisleading\b": 3.5,
        r"\bsuspicious\b": 3.2,
        r"\bunsafe\b": 3.8,
        r"\bfake\b": 3.5,
        r"\bdoes not sound right\b": 3.8,
        r"\bdoesn't sound right\b": 3.8,
    },
    "healthcare_distrust": {
        r"\bwrong diagnosis\b": 4.5,
        r"\bmisdiagnosis\b": 4.8,
        r"\bincorrect diagnosis\b": 4.5,
        r"\bunsafe treatment\b": 4.5,
        r"\bunsafe vaccine\b": 4.0,
        r"\bsecond opinion\b": 3.0,
        r"\bmedical error\b": 4.2,
        r"\bbad advice\b": 3.8,
        r"\bconfusing advice\b": 3.3,
        r"\bdoctor is wrong\b": 4.0,
        r"\bnot medically safe\b": 4.5,
        r"\brisky\b": 3.0,
        r"\bharmful\b": 4.0,
    },
    "uncertainty": {
        r"\bnot sure\b": 4.2,
        r"\bunsure\b": 3.8,
        r"\buncertain\b": 3.8,
        r"\bi am not sure\b": 4.3,
        r"\bi don't know\b": 3.2,
        r"\bi do not know\b": 3.2,
        r"\bhard to say\b": 3.4,
        r"\bunclear\b": 3.2,
        r"\bconfused\b": 3.5,
        r"\bmaybe\b": 2.3,
        r"\bperhaps\b": 2.3,
        r"\bpossibly\b": 2.5,
        r"\bprobably\b": 1.5,
        r"\bseems\b": 1.6,
        r"\bappears\b": 1.6,
        r"\bapparently\b": 2.2,
        r"\bmight\b": 2.4,
        r"\bcould\b": 1.8,
        r"\bmay\b": 1.6,
        r"\bi wonder\b": 2.7,
    },
    "hedging": {
        r"\bi think\b": 2.2,
        r"\bi guess\b": 2.8,
        r"\bi suppose\b": 2.7,
        r"\bit seems\b": 2.2,
        r"\bit appears\b": 2.1,
        r"\bas far as i know\b": 2.0,
        r"\bkind of\b": 1.7,
        r"\bsort of\b": 1.7,
        r"\bnot completely\b": 2.3,
        r"\bnot entirely\b": 2.3,
    },
    "skepticism": {
        r"\bare you sure\b": 3.5,
        r"\bis that true\b": 3.3,
        r"\bprove\b": 2.8,
        r"\bevidence\b": 1.8,
        r"\bsource\b": 1.5,
        r"\bwhere did you get\b": 3.0,
        r"\bhow do you know\b": 3.3,
        r"\bi doubt\b": 3.8,
        r"\bdoubtful\b": 3.5,
        r"\bskeptical\b": 3.5,
    },
    "certainty": {
        r"\bi am sure\b": 3.0,
        r"\bdefinitely\b": 2.8,
        r"\bcertainly\b": 2.6,
        r"\bclearly\b": 2.1,
        r"\babsolutely\b": 2.4,
        r"\bwithout doubt\b": 3.2,
        r"\bno doubt\b": 3.0,
        r"\bconfirmed\b": 2.4,
        r"\bverified\b": 2.5,
    },
    "reassurance": {
        r"\breassured\b": 2.8,
        r"\brelieved\b": 2.0,
        r"\bcomforting\b": 2.2,
        r"\bhelped me\b": 1.8,
        r"\bfeel better\b": 2.0,
        r"\bclear now\b": 2.1,
    }
}


NEGATION_TERMS = ["not", "no", "never", "cannot", "do not", "does not", "did not", "will not", "is not", "without"]

POSITIVE_ANCHORS = ["trust", "trusted", "reliable", "credible", "accurate", "correct", "safe", "helpful", "clear", "reassuring", "confident", "believe", "rely"]


def clean_text(text):
    text = str(text).lower()

    replacements = {
        "don't": "do not",
        "doesn't": "does not",
        "didn't": "did not",
        "can't": "cannot",
        "won't": "will not",
        "isn't": "is not",
        "aren't": "are not",
        "i'm": "i am",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s\.\,\?\!']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def weighted_regex_score(text, pattern_weights):
    total = 0.0
    matches = []

    for pattern, weight in pattern_weights.items():
        found = re.findall(pattern, text)

        if found:
            total += weight * len(found)
            matches.append(pattern.replace(r"\b", "").replace("\\", ""))

    return total, matches


def detect_negated_positive(text, window=4):
    tokens = text.split()
    score = 0.0
    matches = []

    for i, token in enumerate(tokens):
        if token in POSITIVE_ANCHORS:
            start = max(0, i - window)
            context = " ".join(tokens[start:i])

            if any(neg in context for neg in NEGATION_TERMS):
                score += 3.8
                matches.append("negated_" + token)

    return score, matches


def extract_trust_features(text):
    text = clean_text(text)

    output = {}

    for key in LEXICONS:
        output[key], output[key + "_matches"] = weighted_regex_score(text, LEXICONS[key])

    output["negated_positive"], output["negated_positive_matches"] = detect_negated_positive(text)

    output["sentiment"] = SentimentIntensityAnalyzer().polarity_scores(text)["compound"]

    output["confidence_score"] = float(np.clip(
        50 + output["certainty"] * 8 + output["trust_positive"] * 2 - output["uncertainty"] * 8 - output["hedging"] * 6 - output["skepticism"] * 5,
        0,
        100
    ))

    return output


@st.cache_resource
def load_classifier():
    return pipeline(
        "text-classification",
        model=MODEL_DIR,
        tokenizer=MODEL_DIR,
        return_all_scores=True,
        device=0 if torch.cuda.is_available() else -1
    )


classifier = load_classifier()


def entropy_from_probs(probs):
    probs = np.array(probs, dtype=float)
    probs = np.clip(probs, 1e-12, 1.0)
    entropy = -np.sum(probs * np.log(probs))
    return float(entropy / np.log(len(probs)))


def get_roberta_outputs(text):
    # When return_all_scores=True and given a single input, the pipeline
    # returns a list of dictionaries (e.g., [{'label': 'LABEL_0', 'score': 0.9}, ...])
    scores = classifier(text)

    # Create a dictionary mapping labels to scores from this list of dictionaries
    score_dict = {item["label"]: item["score"] for item in scores}

    low = score_dict.get("Low Trust", score_dict.get("LABEL_0", 0.0))
    medium = score_dict.get("Medium Trust", score_dict.get("LABEL_1", 0.0))
    high = score_dict.get("High Trust", score_dict.get("LABEL_2", 0.0))

    probs = np.array([low, medium, high], dtype=float)
    probs = probs / probs.sum()

    entropy = entropy_from_probs(probs)

    return {
        "low_prob": float(probs[0]),
        "medium_prob": float(probs[1]),
        "high_prob": float(probs[2]),
        "entropy": entropy,
        "model_confidence": float(1.0 - entropy),
        "expected_trust": float((probs[2] * 100) + (probs[1] * 50)),
    }


def final_trust_score(text):
    features = extract_trust_features(text)
    roberta = get_roberta_outputs(text)

    baseline = 45.0

    roberta_contribution = (roberta["expected_trust"] - 50) * 0.45
    confidence_contribution = (features["confidence_score"] - 50) * 0.20

    positive_contribution = (
        features["healthcare_trust"] * 4.5 +
        features["trust_positive"] * 3.0 +
        features["certainty"] * 2.0 +
        features["reassurance"] * 2.5
    )

    negative_contribution = (
        features["healthcare_distrust"] * 5.5 +
        features["distrust_negative"] * 5.0 +
        features["uncertainty"] * 4.8 +
        features["hedging"] * 3.8 +
        features["skepticism"] * 4.2 +
        features["negated_positive"] * 5.5
    )

    entropy_penalty = roberta["entropy"] * 12.0
    sentiment_support = features["sentiment"] * 3.0

    score = baseline + roberta_contribution + confidence_contribution + positive_contribution - negative_contribution - entropy_penalty + sentiment_support
    score = float(np.clip(score, 0, 100))

    if score < 35:
        label = "Low Trust"
    elif score < 70:
        label = "Medium Trust"
    else:
        label = "High Trust"

    return score, label, features, roberta, positive_contribution, negative_contribution, entropy_penalty, sentiment_support


st.set_page_config(page_title="Implicit Trust Signal Detection", layout="wide")

st.title("Implicit Trust Signal Detection in Healthcare Chatbot Conversations")

user_text = st.text_area(
    "Enter healthcare chatbot conversation text",
    height=140,
    value="I'm not sure if this trust detection system will work properly."
)

if st.button("Analyze Trust"):

    # Check for empty input
    if not user_text.strip():
        st.warning("Please enter some text before analyzing.")
        st.stop()

    # Run the trust detection model
    score, label, features, roberta, pos, neg, entropy_penalty, sentiment_support = final_trust_score(user_text)

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Trust Score", f"{score:.2f}/100")
    col2.metric("Trust Label", label)
    col3.metric("Model Confidence", f"{roberta['model_confidence']:.3f}")
    col4.metric("Prediction Entropy", f"{roberta['entropy']:.3f}")

    st.subheader("RoBERTa Probabilities")

    prob_df = pd.DataFrame({
        "Label": LABEL_NAMES,
        "Probability": [roberta["low_prob"], roberta["medium_prob"], roberta["high_prob"]]
    })

    st.bar_chart(prob_df.set_index("Label"))

    st.subheader("Detected Trust Signals")

    signal_data = {
        "Healthcare Trust Signals": features["healthcare_trust_matches"],
        "Healthcare Distrust Signals": features["healthcare_distrust_matches"],
        "Trust Signals": features["trust_positive_matches"],
        "Distrust Signals": features["distrust_negative_matches"],
        "Uncertainty Signals": features["uncertainty_matches"],
        "Hedging Signals": features["hedging_matches"],
        "Skepticism Signals": features["skepticism_matches"],
        "Negated Positive Signals": features["negated_positive_matches"],
        "Certainty Signals": features["certainty_matches"],
        "Reassurance Signals": features["reassurance_matches"],
    }

    st.json(signal_data)

    st.subheader("Feature Contribution Chart")

    contribution_df = pd.DataFrame({
        "Feature Group": [
            "Positive trust evidence",
            "Negative trust evidence",
            "Entropy penalty",
            "Sentiment support",
            "Confidence contribution"
        ],
        "Contribution": [
            pos,
            -neg,
            -entropy_penalty,
            sentiment_support,
            (features["confidence_score"] - 50) * 0.20
        ]
    })

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=contribution_df, x="Contribution", y="Feature Group", ax=ax)
    ax.axvline(0, color="black", linewidth=1)
    st.pyplot(fig)

    st.subheader("Feature Table")

    st.dataframe(contribution_df)

In [ ]:
# ============================================================
# CELL 25: Run Streamlit app in Google Colab
# ============================================================

# After running this cell, open the public URL printed by localtunnel
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
!pip install -q streamlit pyngrok transformers vaderSentiment seaborn
!ls

In [ ]:
import subprocess
import time

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)

print("✅ Streamlit started")

In [ ]:
from pyngrok import ngrok

# Stop any tunnels started in this runtime
ngrok.kill()

# Authenticate ngrok with your authtoken
# Replace 'YOUR_NGROK_AUTHTOKEN' with your actual token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3FU7AhE7XNGqeYo4aOqXrM28e87_HALcwgca3VcuU85vYKX1")

# Create a NEW random tunnel
public_url = ngrok.connect(addr=8501)

print(public_url.public_url)